# 03 · Endpoint 배포 & invoke 스모크 — 텍스트 분류(intent)

**TL;DR** — 파인튜닝한 Gemma SLM을 SageMaker real-time endpoint로 배포하고, 실제 호출을 통해 정상 동작을 검증합니다.

**Why** — real-time endpoint는 GPU를 상시 확보한 채 서빙하므로 지연이 낮고, 이후 agentic 루프에서 tool로 바로 호출할 수 있습니다.

**기존 Pain Point** — serverless inference는 GPU를 제공하지 않아 SLM/LLM 서빙에는 적합하지 않으므로, 여기서는 real-time endpoint를 사용합니다.

> 🔴 실제 실행 시 AWS 자격증명·GPU·엔드포인트 과금이 발생합니다. 먼저 `DRY_RUN=1`로 파이프라인을 검증하세요.

In [ ]:
import os, sys
# 리포 루트를 path에 추가해 common/ 와 트랙 로컬 모듈을 import
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, REPO)
sys.path.insert(0, os.getcwd())

In [ ]:
import importlib, boto3
from common import config, aws_utils; importlib.reload(config)
from sagemaker.core.helper.session_helper import Session
sess = Session(boto3.Session(region_name=config.AWS_REGION))
%store -r md_classification
%store -r model_data
%store -r role
model_data = globals().get('md_classification') or globals().get('model_data')
# model_data = 이 트랙의 최신 학습 산출물(SFT 또는 GRPO — 서빙 형식 동일).
# %store 오염 방지: role이 없거나 옛 플레이스홀더면 다시 해석.
if 'role' not in dir() or not role or ':role/' not in str(role):
    role = config.resolve_sagemaker_role(sess)

# 리전 가드: %store 값이 옛 리전을 가리키면 자동 교체(ensure_model_data_in_region 독스트링 참고).
model_data = aws_utils.ensure_model_data_in_region(
    locals().get('model_data'), config.AWS_REGION, job_prefix='gemma-classification-train')
md_classification = model_data
%store model_data
%store md_classification
print('model_data:', model_data, '  ← 이 산출물을 배포합니다')
print('role      :', role)

## 배포 모드 3계층 (SDK v3 `Mode`) — 로컬에서 먼저 검증 후 endpoint
SageMaker SDK v3 `ModelBuilder`는 **같은 코드로 3가지 배포 대상**을 고를 수 있습니다(`mode=` 인자). 클라우드 endpoint를 띄우기 전에 로컬에서 먼저 검증하면 시간·비용을 아낍니다.

| 모드 | 실행 위치 | 용도 | 요구 |
|---|---|---|---|
| `Mode.IN_PROCESS` | 현재 파이썬 프로세스 | 가장 빠른 로직 검증(초경량) | 별도 인프라 없음(백엔드 제약 있음) |
| `Mode.LOCAL_CONTAINER` | 로컬 Docker 컨테이너 | endpoint와 **동일 컨테이너**를 로컬에서 재현 | 로컬 Docker + GPU |
| `Mode.SAGEMAKER_ENDPOINT` | SageMaker (클라우드) | 실제 서빙(기본) | AWS 과금 |

`Mode`는 `from sagemaker.serve.mode.function_pointers import Mode`로 가져오며, 지정하지 않으면 `SAGEMAKER_ENDPOINT`가 기본입니다. 아래 1-A/1-B는 모두 이 클라우드 배포 경로에 해당합니다.


이 kit은 gemma-4를 배포하기 전 검증할 때 SDK 로컬 모드(`IN_PROCESS`/`LOCAL_CONTAINER`)를 기본으로 쓰지 않습니다. `IN_PROCESS`는 내부적으로 `transformers.pipeline`이나 `SentenceTransformer`로만 모델을 올리기 때문에 생성형 LLM인 gemma-4는 로드되지 않고, `LOCAL_CONTAINER`도 vLLM DLC에는 해당 분기가 없어 실행되지 않습니다.

그래서 로컬 검증은 앞서 실행한 **`02b_local_serve`**(로컬 GPU에 `vllm serve`)로 하고, 클라우드 배포는 아래 **1-A(vLLM/SGLang DLC)** 또는 **1-B(DJL LMI)** 로 진행합니다. 각 모드를 실제로 돌려 확인한 근거와 예외 사례가 궁금하다면 [`docs/05_serving_containers.md`의 「SDK v3 배포 모드와 로컬 검증」](../../docs/05_serving_containers.md#sdk-v3-배포-모드와-로컬-검증)를 참고하세요.

## 서빙 엔진 선택 — vLLM(기본) · SGLang · DJL LMI
**전 사이즈(E2B/E4B/12B/26B/31B)가 이 세 경로로 서빙됩니다.** 셋 다 **연속 배칭 + OpenAI 호환(`messages`)** 이라 호출 코드가 완전히 같습니다 — 엔진을 바꿔도 04·05 노트북은 그대로 돕니다.

| SERVING_ENGINE | 컨테이너 | 특징 | 실행 절 |
|---|---|---|---|
| `vllm` (기본) | vLLM DLC | 최신 vLLM, 가장 널리 검증됨 | **1-A** |
| `sglang` | SGLang DLC | RadixAttention(프리픽스 캐시 재사용에 강함) | **1-A** (같은 셀) |
| `lmi` | DJL LMI | AWS 관리형 추상화, `OPTION_*` env | **1-B** |

`.env`의 `SERVING_ENGINE`으로 고르고, 이미지는 `*_IMAGE_URI`로 하드코딩해 뒀습니다.

> 🔴 **E2B/E4B에 있었던 함정 (알아두면 유용)**: E계열은 `num_kv_shared_layers>0`인데, transformers가 KV-shared 레이어에 `k_norm`/`k_proj`/`v_proj` 모듈을 만들지 않아 `save_pretrained` 시 그 텐서가 소실됩니다(E4B 실측 54개). vLLM은 전 레이어에 `k_norm`을 등록하므로 `weights not initialized` ValueError로 죽습니다([vLLM #44788](https://github.com/vllm-project/vllm/issues/44788)). **이 kit의 `train.py`가 저장 직전에 그 텐서를 복원**하므로(연산에 쓰이지 않는 dead weight라 정확도 무해) 지금은 E4B도 vLLM으로 정상 서빙됩니다. 즉 #44788은 "E계열은 vLLM 불가"가 아니라 "transformers가 저장한 체크포인트가 vLLM 불가"입니다. 상세: [`docs/05_serving_containers.md` 「E계열 KV-shared dead weight 복원」](../../docs/05_serving_containers.md#e계열-kv-shared-dead-weight-복원)

In [ ]:
from common import config, dlc
# 기본 'vllm'. 바꾸려면 .env의 SERVING_ENGINE=sglang|lmi (또는 여기서 ENGINE=... 직접 지정).
ENGINE = config.SERVING_ENGINE
print('SERVING_ENGINE:', ENGINE, '| model:', config.DEFAULT_MODEL_ID)
print()
# 엔진별 이미지 URI — .env에 완전 URI로 하드코딩돼 있습니다(*_IMAGE_URI).
#    리전을 옮길 땐 AWS_REGION과 .env의 URI 리전을 함께 바꾸세요.
for name, uri in dlc.serving_image_table(config.AWS_REGION).items():
    print(('→ ' if name == ENGINE else '   ') + f'{name:8s} {uri}')
print()
print({'vllm': '아래 1-A 실행(vLLM DLC)', 'sglang': '아래 1-A 실행(SGLang DLC — 같은 셀)',
       'lmi': '아래 1-B 실행(DJL LMI)'}[ENGINE])
# ⚠️ 태그는 자주 갱신됩니다. 실패하면 현행 태그 확인:
#   aws ecr describe-images --registry-id 763104351884 --repository-name vllm --region <region> \
#     --query 'reverse(sort_by(imageDetails,&imagePushedAt))[:5].imageTags'

### 엔진과 서빙 컨테이너는 레이어가 다릅니다
흔한 오해가 "LMI를 쓰면 vLLM을 못 쓴다"인데, 사실 **DJL LMI는 내부에서 vLLM 엔진을 감싸는 AWS 관리형 컨테이너**입니다(`OPTION_ROLLING_BATCH=vllm`). vLLM DLC는 vLLM을 직접 담은 AWS 컨테이너이고요. 즉 1-A와 1-B는 **같은 엔진을 다른 포장으로** 쓰는 선택입니다.

| 구분 | **1-A. vLLM / SGLang DLC** (기본) | **1-B. DJL LMI** (옵션) |
|---|---|---|
| 설정 방식 | `SM_VLLM_*` / `SM_SGLANG_*` env → CLI 플래그 | `OPTION_*` env (예 `OPTION_ROLLING_BATCH=vllm`) |
| 버전 최신성 | 최신(실측 vLLM 0.25.1/0.26.0) | 번들 vLLM 버전에 종속 |
| 언제 | 최신 모델·최신 엔진 기능 | 관리형 추상화·기존 LMI 자산 재사용 |

🔴 **gemma-4는 vLLM ≥ 0.19 필요** → 기본값이 이를 충족합니다. LMI를 쓸 땐 번들 vLLM이 이 조건을 넘는 최신 태그인지 확인하세요. 태그는 배포 직전 [available_images](https://aws.github.io/deep-learning-containers/reference/available_images/)에서 재확인하고, 배경은 [`docs/05_serving_containers.md`](../../docs/05_serving_containers.md)를 참고하세요.
🔴 **endpoint는 삭제 전까지 시간당 과금됩니다** → 실습 후 `99_cleanup` 필수. **1-A/1-B 중 하나만 실행**하세요.

> **텍스트 vs 멀티모달 서빙**: gemma-4/gemma-3-4b+ 는 멀티모달 base입니다. 학습에서 **텍스트 전용으로 re-export**(config `model_type=*_text`)했다면 그냥 텍스트로 서빙됩니다. re-export 안 한 멀티모달 아티팩트를 **텍스트로만** 쓰려면 `--limit-mm-per-prompt`로 이미지/오디오를 0으로 두세요. 이미지→텍스트 등 멀티모달 태스크는 그대로 멀티모달 서빙합니다.

## 1-A. vLLM / SGLang DLC로 배포 (기본, 권장)
`SERVING_ENGINE`이 `vllm`(기본) 또는 `sglang`일 때 실행합니다 — **둘 다 이 셀이 처리합니다**(OpenAI 호환 서버라 호출 스키마가 같습니다). AWS 독립 컨테이너(`vllm:...` / `sglang:...`)에 학습 모델을 실어 배포합니다. sagemaker SDK v3는
배포를 `ModelBuilder`로 정의합니다(v2의 `Model`/`HuggingFaceModel`은 제거됨). `image_uri`(vLLM DLC) +
`s3_model_data_url`(학습 아티팩트) + `env_vars`(`SM_VLLM_*`)만 주면 됩니다 — passthrough 경로라
`schema_builder`가 필요 없습니다. 모델 가중치는 `SM_VLLM_MODEL=/opt/ml/model`(아티팩트 마운트 경로,
머지 모델이 루트에 있음)로 가리킵니다.
🔴 gemma-4 서빙엔 vLLM ≥ 0.19 필요 → vLLM DLC(실측 0.25.1)가 충족. 태그는 실행 전 available_images에서 재확인.
> **텍스트 전용 re-export 모델**(config `model_type=*_text`)은 그대로 텍스트 서빙됩니다. re-export 안 한
>  멀티모달 아티팩트를 텍스트로만 쓰려면 아래 `SM_VLLM_LIMIT_MM_PER_PROMPT` 주석을 해제하세요.

🔴 **`MAX_NUM_SEQS`/`GPU_MEM_UTIL`을 낮춰 둔 이유** — 24GB GPU(L4)에서 vLLM 기본값은 여유가 거의 없습니다. 실측(이 kit endpoint, vLLM 0.26.0, E4B bf16 14.23 GiB): KV 캐시를 배정한 뒤 남은 여유가 **0.47 GiB**뿐이었습니다. 멀티모달 트랙(05)은 vision tower 때문에 가중치가 1 GiB 더 커서 **같은 설정으로 CUDA OOM이 나 배포가 `Failed`**했습니다.
- `max_num_seqs`(기본 **256**)는 샘플러 logits 버퍼를 `256 × vocab 262,144 × 4B = 256 MiB`로 잡습니다. 실습은 동시 요청이 1~2건이므로 **32**로 낮춰도 손실이 없고, 버퍼는 32 MiB로 줄어듭니다.
- 증상이 `did not pass the ping health check`로만 보여 원인을 찾기 어렵습니다 — 실제 `torch.OutOfMemoryError`는 CloudWatch endpoint 로그에만 남습니다.
> 동시 처리량이 필요하면 `MAX_NUM_SEQS`를 올리되, 그때는 `ml.g6e.2xlarge`(L40S 45GB)처럼 큰 GPU를 쓰세요.

In [ ]:
from common import config, dlc
from sagemaker.serve import ModelBuilder
from sagemaker.serve.mode.function_pointers import Mode
import json, time
# 배포 모드: 로컬에서 먼저 검증하려면 Mode.LOCAL_CONTAINER(로컬 Docker+GPU 필요)로 바꿔 실행,
#    검증되면 Mode.SAGEMAKER_ENDPOINT(기본)로 클라우드 배포. (같은 mb 코드, mode만 교체)
DEPLOY_MODE = Mode.SAGEMAKER_ENDPOINT   # 또는 Mode.LOCAL_CONTAINER (로컬 검증)
# 엔진/이미지는 env가 결정합니다(SERVING_ENGINE, *_IMAGE_URI 또는 *_DLC_VERSION).
#    이 셀은 vllm | sglang 둘 다 처리합니다(둘 다 OpenAI 호환 서버 → 호출 스키마 동일).
assert ENGINE in ('vllm', 'sglang'), (
    f"ENGINE={ENGINE!r} — 이 셀은 vllm/sglang 전용입니다. "
    "'lmi'면 아래 1-B를 실행하세요.")
endpoint_name = f'gemma-classification-{ENGINE}-{int(time.time())}'
serve_image = dlc.resolve_serving_image(config.AWS_REGION, ENGINE)
assert serve_image, f'{ENGINE} 이미지 해석 실패 — env로 지정하세요: ' + dlc.AVAILABLE_IMAGES_URL
print(f'{ENGINE} DLC image:', serve_image, '| mode:', DEPLOY_MODE)
# 모델 경로 '/opt/ml/model' — train.py가 머지 모델을 아티팩트 루트에 저장합니다.
# 엔진별 env 키는 dlc.serving_env()가 관리. max_num_seqs/mem_util은 24GB GPU CUDA OOM 방지(docs/05 「24GB GPU CUDA OOM」).
serve_env = dlc.serving_env(
    ENGINE,
    max_model_len=1024,
    max_num_seqs=32,
    gpu_memory_utilization='0.90',
    hf_token=config.get_serving_hf_token(),
    # 멀티모달 base를 '텍스트로만' 서빙할 때(re-export 안 한 경우) 이미지/오디오 차단:
    # mm_limit=json.dumps({'image': 0, 'audio': 0}),
)
print('serve_env:', serve_env)
mb = ModelBuilder(
    image_uri=serve_image,
    s3_model_data_url=model_data,          # 학습 산출 S3 아티팩트 (v3: model_path는 로컬 경로이므로 사용 금지)
    env_vars=serve_env,
    role_arn=role,
    sagemaker_session=sess,
    instance_type=config.INFER_INSTANCE_TYPE,
    mode=DEPLOY_MODE,                      # SAGEMAKER_ENDPOINT(기본) | LOCAL_CONTAINER(로컬 검증)
)
mb.build()
# wait=False로 비동기 배포 — 셀이 'InService'까지 블로킹하지 않고 바로 반환됩니다.
# endpoint 생성은 GPU 프로비저닝 + 컨테이너 pull + 모델 로드로 수 분~십수 분 걸립니다.
# (LOCAL_CONTAINER 모드면 로컬 Docker에 뜨며, wait 등 일부 인자는 무시될 수 있습니다.)
endpoint = mb.deploy(endpoint_name=endpoint_name, initial_instance_count=1,
                     instance_type=config.INFER_INSTANCE_TYPE, wait=False)
# 트랙 전용 키로도 저장 — 전역 키는 다른 트랙이 덮어씁니다(docs/05 「%store 전역 오염」).
ep_classification = endpoint_name
%store endpoint_name
%store ep_classification
from IPython.display import display
print('deploying endpoint:', endpoint_name)
display(aws_utils.cw_links(config.AWS_REGION, endpoint_name=endpoint_name))

### 배포 상태 확인 · 세션이 끊겼을 때 다시 붙기 (재접속)
endpoint 생성도 학습 잡처럼 **SageMaker 서버에서 진행되므로, 커널이나 세션이 끊겨도 계속됩니다.** 위 `deploy(wait=False)`가 바로 반환되니, 아래 셀을 반복 실행해 `Creating → InService` 진행을 확인하세요. 세션이 끊긴 뒤에는 `endpoint` 객체 없이 **endpoint 이름으로 재조회**합니다(v3: `sagemaker.core.resources.Endpoint.get(name)`).
> 이 셀은 커널 재시작 후 위 설정 셀(임포트·`%store -r endpoint_name`)만 실행한 상태에서 바로 쓸 수 있습니다.

In [ ]:
from sagemaker.core.resources import Endpoint
# 트랙 전용 키 우선 — 전역 endpoint_name 은 다른 트랙이 덮어씁니다.
%store -r ep_classification
%store -r endpoint_name
endpoint_name = globals().get('ep_classification') or globals().get('endpoint_name')
assert endpoint_name, 'endpoint_name 이 없습니다 — 03의 배포 셀을 먼저 실행하세요.'
print('사용할 endpoint:', endpoint_name)
ep = Endpoint.get(endpoint_name)
ep.refresh()
print('endpoint:', endpoint_name, '->', ep.endpoint_status)  # Creating / InService / Failed
if ep.endpoint_status == 'Failed':
    print('FailureReason:', getattr(ep, 'failure_reason', None))
# InService까지 폴링 대기하려면 아래 주석을 해제(끊겨도 서버 배포는 계속됨):
# ep.wait_for_status(target_status='InService')
from IPython.display import display
display(aws_utils.cw_links(config.AWS_REGION, endpoint_name=endpoint_name))

## 1-B. (옵션) DJL LMI 컨테이너로 배포 — `SERVING_ENGINE=lmi`
AWS 관리형 서빙 추상화(자동 배칭·라우팅 등)를 원하거나 기존 LMI 자산을 재사용한다면 DJL LMI를 씁니다.
LMI는 내부에서 vLLM 등 백엔드를 감싸며, `OPTION_*` env로 설정합니다(`OPTION_ROLLING_BATCH=vllm`).
🔴 **버전 주의**: gemma-4 서빙엔 vLLM ≥ 0.19가 필요합니다. 번들 vLLM이 낮은 LMI 태그는 gemma-4를
로드하지 못하므로 **최신 태그**를 쓰세요(ECR 실조회 2026-07-30: `0.36.0-lmi27.0.0-cu130-v1.1`이 최신). `LMI_VERSION` 또는 `LMI_IMAGE_URI` env로 지정합니다.
확실한 쪽을 원하면 기본 경로인 1-A(vLLM DLC)를 쓰세요. 1-A를 이미 배포했다면 이 셀은 건너뜁니다(중복 endpoint = 중복 과금).

In [ ]:
RUN_LMI = (ENGINE == 'lmi')   # SERVING_ENGINE=lmi 이면 자동 실행. 강제하려면 True로.
if RUN_LMI:
    from common import dlc
    from sagemaker.serve import ModelBuilder
    import time
    endpoint_name = f'gemma-classification-lmi-{int(time.time())}'
    lmi_image = dlc.resolve_serving_image(config.AWS_REGION, 'lmi')   # env LMI_IMAGE_URI/LMI_VERSION 존중
    print('LMI image:', lmi_image)
    # 1-A와 같은 함수 — 엔진만 'lmi'로 주면 OPTION_* 키로 변환됩니다.
    lmi_env = dlc.serving_env(
        'lmi',
        max_model_len=1024,
        max_num_seqs=32, gpu_memory_utilization='0.90',
        hf_token=config.get_serving_hf_token(),
    )
    print('lmi_env:', lmi_env)
    mb = ModelBuilder(image_uri=lmi_image, s3_model_data_url=model_data,
                      env_vars=lmi_env, role_arn=role, sagemaker_session=sess,
                      instance_type=config.INFER_INSTANCE_TYPE)
    mb.build()
    endpoint = mb.deploy(endpoint_name=endpoint_name, initial_instance_count=1,
                         instance_type=config.INFER_INSTANCE_TYPE, wait=False)  # 비동기
    %store endpoint_name
    from IPython.display import display
    print('deploying endpoint:', endpoint_name)
    display(aws_utils.cw_links(config.AWS_REGION, endpoint_name=endpoint_name))
    # 상태 확인/재접속은 1-A 뒤의 재접속 셀과 동일: Endpoint.get(endpoint_name).refresh()
else:
    print('DJL LMI(B) skipped. Using vLLM DLC endpoint from step 1-A.')

### ⏸️ 세션이 끊겼다면 — 여기서부터 이어서 실행
커널을 재시작했거나 세션이 끊겼다면 위 배포 셀을 다시 돌릴 필요가 없습니다(**endpoint는 서버에 그대로 살아 있습니다**). 아래 셀 하나만 실행하면 호출에 필요한 것(경로·import·`endpoint_name`)이 모두 복구됩니다.
> 이미 위에서부터 순서대로 실행했다면 이 셀은 건너뛰어도 되고, 실행해도 무해합니다.

In [ ]:
# ── 세션 재개 전용 (이 셀만 실행하면 아래 호출 셀들이 바로 동작) ──
import os, sys, importlib
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
for p in (REPO, os.getcwd()):
    if p not in sys.path:
        sys.path.insert(0, p)
from common import config, aws_utils; importlib.reload(config)

# 트랙 전용 키 우선 — 전역 endpoint_name 은 다른 트랙이 덮어씁니다.
%store -r ep_classification
%store -r endpoint_name
endpoint_name = globals().get('ep_classification') or globals().get('endpoint_name')
assert endpoint_name, (
    'endpoint_name 이 없습니다. §1에서 배포하거나, 아래처럼 직접 지정하세요:\n'
    "    endpoint_name = 'gemma-classification-vllm-...'")

from sagemaker.core.resources import Endpoint
ep = Endpoint.get(endpoint_name); ep.refresh()
print('endpoint:', endpoint_name, '->', ep.endpoint_status)
assert ep.endpoint_status == 'InService', (
    f'{ep.endpoint_status} 상태입니다. Creating이면 잠시 뒤 다시 실행하세요.')

## 2. invoke 스모크 (sagemaker-runtime — Bedrock 아님)
배포한 endpoint가 실제로 응답하는지 최소 호출로 확인합니다. 여기서 호출하는 대상은 우리가 배포한 SageMaker endpoint이므로 `sagemaker-runtime` API를 사용합니다(Bedrock 호출이 아닙니다).
위에서 `wait=False`로 배포했으므로, **먼저 endpoint가 `InService`인지 확인**합니다. 아직 `Creating`이면 `InService`가 될 때까지 기다립니다(끊겨도 배포는 서버에서 계속됨).
vLLM · SGLang · LMI(vLLM 백엔드) **셋 다 OpenAI 호환 chat 스키마(`messages`)** 를 받으므로, `invoke_sagemaker_chat`으로 messages를 그대로 보냅니다 — 🔴 **서버가 chat template을 적용**하므로 우리가 렌더할 필요가 없습니다(raw 텍스트를 보내면 template이 빠져 반복·저품질 출력이 납니다. 실측 확인). 응답 파서는 `{"choices":[...]}`와 `{"generated_text"}` 양쪽을 모두 처리합니다.

In [ ]:
# invoke 전 InService 보장 (wait=False 배포이므로)
from sagemaker.core.resources import Endpoint
ep = Endpoint.get(endpoint_name); ep.refresh()
if ep.endpoint_status != 'InService':
    print('waiting for InService (current:', ep.endpoint_status, ')...')
    ep.wait_for_status(target_status='InService')
print('endpoint InService:', endpoint_name)

In [ ]:
import importlib, track_data as td; importlib.reload(td)
user = "My new card hasn't arrived yet, what should I do?"
messages = [{'role': 'user', 'content': f'{td.SYSTEM_PROMPT}\n\n{user}'}]
# vllm/sglang/lmi 모두 messages 스키마 → 서버가 chat template을 적용합니다.
out = aws_utils.invoke_sagemaker_chat(endpoint_name, messages, region=config.AWS_REGION,
                                     max_tokens=256, temperature=0.1)
from common.display_utils import show_inference
show_inference(user, out, title='배포 스모크')

## 3. 실시간 추론 (실제 서비스와 동일한 호출)
실제 배포 환경에는 정답(reference)이 없습니다 — 새 입력이 들어오면 그대로 추론해 응답할 뿐입니다. 여기서는 **학습에 쓰지 않은 입력 1~2건**으로 실시간 추론을 보여 줍니다. (학습은 데이터셋 앞부분만 사용하므로, 그 뒤 슬라이스에서 입력을 뽑아 '새 입력' 상황을 재현합니다.)
> 정답과 비교해 성능을 수치로 재는 held-out 평가는 뒤의 **evaluate 노트북**에서 다룹니다.

**스트리밍(`STREAM`)** — vLLM/SGLang/LMI는 OpenAI 호환 SSE를 지원하므로 `invoke_endpoint_with_response_stream`으로 **토큰이 생성되는 대로** 받아볼 수 있습니다. 실측(요약 트랙, vLLM 0.26.0): **첫 응답 0.42초 vs 완성 대기 16.16초 → 체감 38배**.
이 트랙은 응답이 JSON/라벨이라 완성돼야 쓸 수 있으므로 기본은 끕니다(켜도 동작합니다).
> 🔴 스트리밍은 **첫 토큰 체감만** 줄입니다 — 전체 생성 시간이나 동시 처리량(throughput)은 그대로입니다. 위 실측에서도 완료 시각은 15.9s vs 16.2s로 사실상 같습니다.

In [ ]:
import importlib, track_data as td; importlib.reload(td)
# 학습에 안 쓴 입력(앞부분 NUM_SEED_SAMPLES건 이후)에서 2건만 로드 — 입력만 사용.
holdout = td.load_seed_examples(config.NUM_SEED_SAMPLES + 2, token=config.get_hf_token())[-2:]

from common.display_utils import show_inference, stream_inference

# STREAM=True: 토큰을 생성되는 대로 표시(docs/05 「응답 스트리밍」).
STREAM = False

def msgs_for(user_input: str) -> list:
    """vllm/sglang/lmi 공통 chat 스키마 — 서버가 chat template을 적용합니다."""
    return [{'role': 'user', 'content': f'{td.SYSTEM_PROMPT}\n\n{user_input}'}]

# 마크다운 렌더 — 긴 입력은 접고 JSON은 들여쓰기(print는 절단됨).
for i, ex in enumerate(holdout, 1):
    if STREAM:
        pieces = aws_utils.stream_sagemaker_chat(
            endpoint_name, msgs_for(ex['input']), region=config.AWS_REGION,
            max_tokens=256, temperature=0.2)
        stream_inference(ex['input'], pieces, index=i)
    else:
        out = aws_utils.invoke_sagemaker_chat(
            endpoint_name, msgs_for(ex['input']), region=config.AWS_REGION,
            max_tokens=256, temperature=0.2)
        show_inference(ex['input'], out, index=i)

## 4. (선택) LiteLLM 게이트웨이로도 호출
여러 프로바이더를 하나의 인터페이스로 다루고 싶다면 LiteLLM 게이트웨이를 통해서도 endpoint를 호출할 수 있습니다. 다만 LiteLLM은 sagemaker 패키지와 importlib-metadata 의존성이 충돌해 코어 의존성에 포함하지 않았으므로, 필요할 때만 `pip install 'litellm>=1.93.0'`으로 별도 설치합니다.
⚠️ 핵심 경로는 위 2번(sagemaker-runtime)만으로 완결되며, 이 셀은 통합 인터페이스를 보여 주기 위한 선택적 데모입니다.

In [ ]:
try:
    from common import llm_gateway as gw
    resp = gw.endpoint_chat(user, endpoint_name, region=config.AWS_REGION,
                            chat_route=False, hf_model_name=config.DEFAULT_MODEL_ID, max_tokens=256)
    print('via LiteLLM:\n', resp)
except ImportError:
    print('litellm not installed - optional. To use: pip install \'litellm>=1.93.0\' (separate env recommended)')
except Exception as e:
    print('LiteLLM path needs tuning to the endpoint serving schema:', e)

✅ endpoint 동작을 확인했습니다. 다음은 **04_evaluate.ipynb**로 held-out 성능을 수치로 확인합니다. (⚠️ 실습이 끝나면 endpoint를 반드시 삭제하세요.)